<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/ai-audio/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q allosaurus torchcrepe librosa soundfile scipy numpy requests

import os, json, zipfile, requests, unicodedata
import numpy as np
import torch
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks
from allosaurus.app import read_recognizer

os.makedirs("audio/phonemes", exist_ok=True)
os.makedirs("audio/slices", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
recognizer = read_recognizer()

# 1. Canonical Clean IPA Phoneme Bank (Wikimedia Commons / UCLA Phonetics Lab)
CANONICAL_PHONEMES = {
    # Consonants
    "k":  ("k.wav",  "https://upload.wikimedia.org/wikipedia/commons/e/e3/Voiceless_velar_plosive.ogg"),
    "kʰ": ("k.wav",  "https://upload.wikimedia.org/wikipedia/commons/e/e3/Voiceless_velar_plosive.ogg"),
    "t":  ("t.wav",  "https://upload.wikimedia.org/wikipedia/commons/0/02/Voiceless_alveolar_plosive.ogg"),
    "tʰ": ("t.wav",  "https://upload.wikimedia.org/wikipedia/commons/0/02/Voiceless_alveolar_plosive.ogg"),
    "p":  ("p.wav",  "https://upload.wikimedia.org/wikipedia/commons/5/51/Voiceless_bilabial_plosive.ogg"),
    "d":  ("d.wav",  "https://upload.wikimedia.org/wikipedia/commons/0/01/Voiced_alveolar_plosive.ogg"),
    "ɡ":  ("g.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/12/Voiced_velar_plosive_02.ogg"),
    "g":  ("g.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/12/Voiced_velar_plosive_02.ogg"),
    "dʒ": ("dz.wav", "https://upload.wikimedia.org/wikipedia/commons/e/e6/Voiced_palato-alveolar_affricate.ogg"),
    "w":  ("w.wav",  "https://upload.wikimedia.org/wikipedia/commons/f/f2/Voiced_labio-velar_approximant.ogg"),
    "l":  ("l.wav",  "https://upload.wikimedia.org/wikipedia/commons/b/bc/Alveolar_lateral_approximant.ogg"),
    "r":  ("r.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/1f/Alveolar_approximant.ogg"),
    "ɹ":  ("r.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/1f/Alveolar_approximant.ogg"),
    "m":  ("m.wav",  "https://upload.wikimedia.org/wikipedia/commons/a/a9/Bilabial_nasal.ogg"),
    "n":  ("n.wav",  "https://upload.wikimedia.org/wikipedia/commons/2/29/Alveolar_nasal.ogg"),
    # Vowels
    "æ":  ("ae.wav", "https://upload.wikimedia.org/wikipedia/commons/c/c9/Near-open_front_unrounded_vowel.ogg"),
    "ɑ":  ("aa.wav", "https://upload.wikimedia.org/wikipedia/commons/e/e5/Open_back_unrounded_vowel.ogg"),
    "ɔ":  ("ao.wav", "https://upload.wikimedia.org/wikipedia/commons/d/d0/PR-open-mid_back_rounded_vowel.ogg"),
    "ʌ":  ("ah.wav", "https://upload.wikimedia.org/wikipedia/commons/9/92/Open-mid_back_unrounded_vowel.ogg"),
    "i":  ("iy.wav", "https://upload.wikimedia.org/wikipedia/commons/9/91/Close_front_unrounded_vowel.ogg"),
    "iː": ("iy.wav", "https://upload.wikimedia.org/wikipedia/commons/9/91/Close_front_unrounded_vowel.ogg"),
}

IPA_BASE_VOWELS = set("aeiouyɨʉɯuɪʏʊeøɘɵɤoəɛœɜɞʌɔæɐaɶɑɒɚɝ")
HEADERS = {"User-Agent": "GroundLevelLanguages/1.0 (Academic Research Prototype)"}

def classify_phone(ipa_symbol):
    decomposed = unicodedata.normalize("NFD", ipa_symbol)
    base_chars = [ch for ch in decomposed if unicodedata.category(ch) != "Mn"]
    return "vowel" if any(ch in IPA_BASE_VOWELS for ch in base_chars) else "consonant"

def download_canonical_bank():
    downloaded = set()
    for ipa, (fname, url) in CANONICAL_PHONEMES.items():
        out_path = f"audio/phonemes/{fname}"
        if fname in downloaded or os.path.exists(out_path):
            continue
        r = requests.get(url, headers=HEADERS)
        if r.status_code == 200:
            tmp_ogg = f"/content/tmp_{fname}.ogg"
            with open(tmp_ogg, "wb") as f:
                f.write(r.content)
            y_clip, _ = librosa.load(tmp_ogg, sr=16000)
            y_clip, _ = librosa.effects.trim(y_clip, top_db=30)
            sf.write(out_path, y_clip, 16000)
            downloaded.add(fname)
            print(f"Saved clean phoneme: {out_path}")

def apply_fade(clip, sr=16000, fade_ms=10):
    fade_len = min(int(sr * (fade_ms / 1000.0)), len(clip) // 2)
    if fade_len <= 1:
        return clip
    win = np.ones(len(clip), dtype=np.float32)
    win[:fade_len] = np.linspace(0.0, 1.0, fade_len)
    win[-fade_len:] = np.linspace(1.0, 0.0, fade_len)
    return clip * win

def synthesize_humming_track(y, f0, periodicity, sr=16000, hop_length=160):
    total_samples = len(y)
    freq_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(f0)), f0)
    conf_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(periodicity)), periodicity)
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=hop_length)[0]
    rms_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(rms)), rms)
    rms_norm = rms_interp / (np.max(rms_interp) + 1e-6)
    voiced_mask = (conf_interp > 0.15) & (rms_norm > 0.12)
    amp = np.where(voiced_mask, 0.35 * rms_norm, 0.0)
    phase = (2.0 * np.pi * np.cumsum(freq_interp)) / sr
    return (amp * (0.8 * np.sin(phase) + 0.2 * np.sin(2.0 * phase))).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    y_raw, sr = librosa.load(mp3_path, sr=16000)
    y, _ = librosa.effects.trim(y_raw, top_db=28)
    y = np.pad(y, (int(0.05 * sr), int(0.05 * sr)), mode="constant")
    temp_wav = f"/content/{word_id}_16k.wav"
    sf.write(temp_wav, y, sr)

    # STEP 1: Detect phonemes with Allosaurus & map to clean canonical recordings
    raw_phones = recognizer.recognize(temp_wav, lang_id='eng', timestamp=False, topk=1).split()
    phonemes_out = []
    for idx, ipa in enumerate(raw_phones):
        clean_base = unicodedata.normalize("NFD", ipa)[0]
        fname, _ = CANONICAL_PHONEMES.get(ipa, CANONICAL_PHONEMES.get(clean_base, ("ah.wav", "")))
        phonemes_out.append({
            "id": f"{word_id}_p{idx+1}",
            "kind": classify_phone(ipa),
            "audio_url": f"audio/phonemes/{fname}"
        })

    print(f"[{word_id}] Detected {len(phonemes_out)} phonemes -> {raw_phones}")

    # STEP 2: Syllable segmentation
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.2)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]
    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s, e_s = (boundaries[idx] * 160) / sr, (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        sf.write(clip_path, apply_fade(y[int(s_s * sr):int(e_s * sr)], sr=sr), sr)
        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(65, dur * 240))),
            "audio_url": clip_path
        })

    # STEP 3: TorchCREPE F0 humming
    pitch, periodicity = torchcrepe.predict(
        torch.tensor(y).unsqueeze(0), sr, hop_length=160, fmin=65.0, fmax=500.0,
        model="full", decoder=torchcrepe.decode.viterbi, return_periodicity=True, device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()
    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(y, frequency, conf, sr=sr), sr)

    tones_out = []
    for idx in range(len(syllables_out)):
        s_f = int((idx / len(syllables_out)) * len(frequency))
        e_f = int(((idx + 1) / len(syllables_out)) * len(frequency))
        slope = float(frequency[e_f-1] - frequency[s_f]) if e_f - s_f > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({"id": f"{word_id}_t{idx+1}", "style": style, "audio_url": hum_out})

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

download_canonical_bank()

words = [
    ("cat", "/content/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "/content/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "/content/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "/content/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "/content/tree.mp3", "images/visuals/tree.jpg")
]

dataset = {"items": [process_word(w, p, v) for w, p, v in words if os.path.exists(p)]}
with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

with zipfile.ZipFile("gll_update2_assets.zip", "w") as zf:
    zf.write("lesson_data.json")
    for folder in ["audio/phonemes", "audio/slices"]:
        for root, _, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))

print("Done! Extract gll_update2_assets.zip into your project root.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
downloading model  latest
from:  https://github.com/xinjli/allosaurus/releases/download/v1.0/latest.tar.gz
to:    /usr/local/lib/python3.13/dist-packages/allosaurus/pretrained
please wait...
Saved clean phoneme: audio/phonemes/t.wav
Saved clean phoneme: audio/phonemes/d.wav
Saved clean phoneme: audio/phonemes/g.wav
Saved clean phoneme: audio/phonemes/m.wav
Done! Extract gll_update2_assets.zip into your project root.
